# Projeto: Detecção de Fraude em Planos de Saúde

## 1. Contexto do Problema
A fraude na saúde representa de **3% a 10%** dos gastos totais do setor. Este dataset simula 10.000 reivindicações (claims) reais, onde o comportamento fraudulento muitas vezes se camufla em atividades médicas legítimas.

---

## 2. Dicionário de Dados

### Informações Financeiras e do Claim
* `Claim_Amount`: Valor solicitado.
* `Approved_Amount`: Valor aprovado pelo plano.
* `Claim_Status`: Status da solicitação (Aprovada, Rejeitada, Pendente).
* `Insurance_Type`: Tipo de seguro (Private, Medicare, Medicaid, Self-pay).

### Informações do Paciente
* `Patient_Age` / `Gender`: Idade e gênero.
* `State`: Localização geográfica.
* `Chronic_Condition`: Indicador de condição crônica.
* `Prior_Visit_History`: Histórico de visitas nos últimos 12 meses.

### Informações do Provedor (Médico/Hospital)
* `Provider_ID`: Identificador único do provedor.
* `Provider_Specialty`: Especialidade (Cardiologia, Ortopedia, Clínica Geral, etc.).
* `Monthly_Claim_Volume`: Volume mensal de solicitações por provedor.

### Codificação Médica
* `Diagnosis_Codes`: Códigos ICD-10 de diagnóstico.
* `Procedure_Codes`: Códigos CPT de procedimentos.

### Atributos Temporais e Adicionais
* `Claim_Submission_Date`: Data de submissão.
* `Days_to_Submission`: Dias entre o serviço e a submissão da cobrança.
* `Length_of_Stay`: Duração da internação (para casos Inpatient).
* `Visit_Type`: Tipo de visita (Outpatient, Inpatient, Emergency).

---

## 3. Variável Alvo (Target)
* **`Is_Fraud`**: 
    * `0`: Reivindicação Legítima.
    * `1`: Reivindicação Fraudulenta.
* **Taxa de Fraude**: Aproximadamente **8%** (Classe desbalanceada).

---

## 4. Padrões de Fraude Esperados
1. Reivindicações fraudulentas tendem a ter **valores mais altos**.
2. Casos de fraude costumam ser submetidos com **menor atraso** após o serviço.
3. Certos provedores concentram maior atividade suspeita.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#removendo os limites de visualização
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
pd.options.display.float_format = '{:.2f}'.format


In [3]:
#importando o meu dataset
path = r"C:\Users\User\Desktop\projetos\projeto-fraude-saude\data\raw-data\healthcare_fraud_detection.csv"
df = pd.read_csv(path)

## 1 Avaliando o conjunto de dados de forma macro para compreender os dados

In [4]:
df.head()

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.00
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.00
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.00
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.00
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.00


In [5]:
#Quantas linhas e colunas tem no dataframe
df.shape

(10000, 20)

In [6]:
#Tipos de dado de cada coluna
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Provider_ID                            10000 non-null  object 
 1   Claim_ID                               10000 non-null  object 
 2   Patient_Age                            10000 non-null  int64  
 3   Patient_Gender                         10000 non-null  object 
 4   Diagnosis_Code                         10000 non-null  object 
 5   Procedure_Code                         10000 non-null  int64  
 6   Claim_Amount                           10000 non-null  float64
 7   Approved_Amount                        10000 non-null  float64
 8   Insurance_Type                         9650 non-null   object 
 9   Claim_Submission_Date                  10000 non-null  object 
 10  Days_Between_Service_and_Claim         10000 non-null  int64  
 11  Num

## 2 Verificar se existem valores nulos nos dados


In [7]:
df.isnull().sum()

Provider_ID                                0
Claim_ID                                   0
Patient_Age                                0
Patient_Gender                             0
Diagnosis_Code                             0
Procedure_Code                             0
Claim_Amount                               0
Approved_Amount                            0
Insurance_Type                           350
Claim_Submission_Date                      0
Days_Between_Service_and_Claim             0
Number_of_Claims_Per_Provider_Monthly      0
Provider_Specialty                       350
Patient_State                              0
Claim_Status                               0
Is_Fraud                                   0
Length_of_Stay                             0
Visit_Type                                 0
Chronic_Condition_Flag                     0
Prior_Visits_12m                         350
dtype: int64

Insight de Engenharia de Dados: O Padrão dos Nulos Coincidentes

Aqui temos a primeira informação importante do projeto. As colunas Insurance_Type, Provider_Specialty, Prior_Visits_12m tem exatamente 350 valores nulos e isso não parece ser coincidência. Geralmente quando o número nulo "bate" perfeitamente entre colunas, isso indica um padrão sistêmico.

As colunas que estão nulas são:

* `Insurance_Type` (Tipo de Seguro)
* `Provider_Specialty` (Especialidade do Médico)
* `Prior_Visits_12m` (Histórico de Visitas)

Fraudadores muitas vezes tentam criar "contas fantasma" ou registrar procedimentos para pacientes que não existem ou que não frequentam aquela clínica. A falta de especialidade do provedor ou do histórico de visitas do paciente pode indicar que esses registros foram gerados artificialmente no sistema de cobrança, sem uma base de dados real por trás para preencher esses campos automáticos.

In [8]:
# Tratando as colunas de texto (Categorias)
df['Provider_Specialty'] = df['Provider_Specialty'].fillna('UNKNOWN')
df['Insurance_Type'] = df['Insurance_Type'].fillna('UNKNOWN')

# Tratando a coluna numérica
# Aqui, em vez de UNKNOWN (que é texto), usamos -1 para manter a coluna como número
# mas indicar que o valor original era inexistente.
df['Prior_Visits_12m'] = df['Prior_Visits_12m'].fillna(-1)

# Verificando se ainda resta algum nulo
print(df.isnull().sum())

Provider_ID                              0
Claim_ID                                 0
Patient_Age                              0
Patient_Gender                           0
Diagnosis_Code                           0
Procedure_Code                           0
Claim_Amount                             0
Approved_Amount                          0
Insurance_Type                           0
Claim_Submission_Date                    0
Days_Between_Service_and_Claim           0
Number_of_Claims_Per_Provider_Monthly    0
Provider_Specialty                       0
Patient_State                            0
Claim_Status                             0
Is_Fraud                                 0
Length_of_Stay                           0
Visit_Type                               0
Chronic_Condition_Flag                   0
Prior_Visits_12m                         0
dtype: int64


## 3 Verificar os valores únicos em cada variável

In [9]:
# Mostra a quantidade de valores únicos por coluna
df.nunique()

Provider_ID                                300
Claim_ID                                 10000
Patient_Age                                 95
Patient_Gender                               2
Diagnosis_Code                              10
Procedure_Code                               9
Claim_Amount                              9460
Approved_Amount                           9373
Insurance_Type                               5
Claim_Submission_Date                     1476
Days_Between_Service_and_Claim              30
Number_of_Claims_Per_Provider_Monthly      100
Provider_Specialty                           7
Patient_State                                8
Claim_Status                                 3
Is_Fraud                                     2
Length_of_Stay                               5
Visit_Type                                   3
Chronic_Condition_Flag                       2
Prior_Visits_12m                            14
dtype: int64

In [10]:
# Lista de colunas que vamos descartar
colunas_para_remover = ['Provider_ID', 'Claim_ID','Claim_Status']

# Removendo e salvando no mesmo dataframe
df = df.drop(columns=colunas_para_remover, axis=1)

# Verificando se elas sumiram
print(f"Colunas restantes: {df.columns.tolist()}")

Colunas restantes: ['Patient_Age', 'Patient_Gender', 'Diagnosis_Code', 'Procedure_Code', 'Claim_Amount', 'Approved_Amount', 'Insurance_Type', 'Claim_Submission_Date', 'Days_Between_Service_and_Claim', 'Number_of_Claims_Per_Provider_Monthly', 'Provider_Specialty', 'Patient_State', 'Is_Fraud', 'Length_of_Stay', 'Visit_Type', 'Chronic_Condition_Flag', 'Prior_Visits_12m']


In [11]:
# O número de datas estaca muito alto, isso é péssimo para o modelo, então vamos tratar.

# 1. Convertendo para datetime
df['Claim_Submission_Date'] = pd.to_datetime(df['Claim_Submission_Date'])

# 2. Extraindo características numéricas
df['Submission_Month'] = df['Claim_Submission_Date'].dt.month
df['Submission_DayOfWeek'] = df['Claim_Submission_Date'].dt.dayofweek  # 0=Segunda, 6=Domingo

# 3. Agora podemos remover a coluna original de data
df = df.drop(columns=['Claim_Submission_Date'])

# Visualizando como ficou
print(df[['Submission_Month', 'Submission_DayOfWeek']].head())

   Submission_Month  Submission_DayOfWeek
0                 9                     6
1                 9                     0
2                 4                     0
3                10                     2
4                 9                     1


One Hot encoding

Para treinar o modelo vamos utilizar a regressão logística, e para isso precisamos transformar os dados categoricos (strings) em dados numéricos. Agora vamos transformar os dados categoricos em 0 e 1.

In [12]:
# Lista das colunas que ainda são texto (Categorias)
colunas_categoricas = [
    'Patient_Gender', 'Insurance_Type', 'Provider_Specialty', 
    'Patient_State', 'Visit_Type', 'Diagnosis_Code', 'Procedure_Code'
]

# Transformando em 0 e 1
# O argumento drop_first=True é uma boa prática para Regressão Logística 
# para evitar a multicolinearidade (um problema matemático do modelo)
df_final = pd.get_dummies(df, columns=colunas_categoricas, drop_first=True)

# Verificando o resultado
print(f"Total de colunas após o encoding: {df_final.shape[1]}")
df_final.head()

Total de colunas após o encoding: 48


,Patient_Age,Claim_Amount,Approved_Amount,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Is_Fraud,Length_of_Stay,Chronic_Condition_Flag,Prior_Visits_12m,Submission_Month,Submission_DayOfWeek,Patient_Gender_Male,Insurance_Type_Medicare,Insurance_Type_Private,Insurance_Type_Self-Pay,Insurance_Type_UNKNOWN,Provider_Specialty_General Practice,Provider_Specialty_Internal Medicine,Provider_Specialty_Neurology,Provider_Specialty_Orthopedics,Provider_Specialty_Pulmonology,Provider_Specialty_UNKNOWN,Patient_State_FL,Patient_State_GA,Patient_State_IL,Patient_State_NY,Patient_State_OH,Patient_State_PA,Patient_State_TX,Visit_Type_Inpatient,Visit_Type_Outpatient,Diagnosis_Code_E78.5,Diagnosis_Code_F41.9,Diagnosis_Code_I10,Diagnosis_Code_I25.10,Diagnosis_Code_J06.9,Diagnosis_Code_J18.9,Diagnosis_Code_K21.9,Diagnosis_Code_M54.5,Diagnosis_Code_N39.0,Procedure_Code_71046,Procedure_Code_80053,Procedure_Code_85025,Procedure_Code_87086,Procedure_Code_93000,Procedure_Code_97110,Procedure_Code_99213,Procedure_Code_99214
0,37,443.51,393.16,13,70,0,0,1,2.00,9,6,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,21,467.50,461.33,5,62,0,5,1,2.00,9,0,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,78,591.69,530.06,29,60,0,5,1,3.00,4,0,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False
3,65,235.15,189.11,22,70,0,0,0,5.00,10,2,True,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False
4,36,487.96,369.91,21,67,0,5,0,4.00,9,1,True,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False


In [13]:
# Mostra quantas vezes o número '1' aparece em cada uma das novas colunas
colunas_novas = df_final.select_dtypes(include=['uint8', 'bool']).columns
print(df_final[colunas_novas].sum().sort_values())

Insurance_Type_UNKNOWN                   350
Provider_Specialty_UNKNOWN               350
Procedure_Code_99214                     531
Procedure_Code_80053                     915
Procedure_Code_85025                     937
Diagnosis_Code_J18.9                     943
Diagnosis_Code_I10                       951
Diagnosis_Code_J06.9                     955
Diagnosis_Code_F41.9                     998
Diagnosis_Code_K21.9                    1004
Diagnosis_Code_N39.0                    1016
Diagnosis_Code_I25.10                   1025
Diagnosis_Code_E78.5                    1027
Diagnosis_Code_M54.5                    1055
Patient_State_NY                        1187
Procedure_Code_71046                    1208
Patient_State_OH                        1225
Procedure_Code_93000                    1234
Patient_State_IL                        1245
Patient_State_FL                        1245
Patient_State_GA                        1252
Provider_Specialty_Pulmonology          1255
Procedure_

In [ ]:
import os

# 1. Criar a pasta 'processed' caso ela ainda não exista
output_dir = "../data/processed"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Definir o nome do ficheiro final
file_path = os.path.join(output_dir, "healthcare_fraud_final_model.csv")

# 3. Salvar o dataframe (df_final) para CSV
# index=False evita que o Pandas crie uma coluna de números desnecessária
df_final.to_csv(file_path, index=False)

print(f"Sucesso! O dataset com {df_final.shape[1]} colunas foi salvo em: {file_path}")

Sucesso! O dataset com 48 colunas foi salvo em: ../data/processed\healthcare_fraud_final_model.csv
